# Étape 2 — Nettoyage et préparation des tables pour l'app

**Entrée :** `conso_regionale.csv` et `conso_nationale.csv`, téléchargés à l'étape 1.

**Sorties :** des petites tables, quelques Mo au lieu de 300, que l'app Streamlit chargera rapidement :

| Fichier | Contenu |
|---|---|
| `conso_jour_region.csv` | Gaz et électricité par jour et par région, en GWh |
| `profil_horaire.csv` | Puissance moyenne par heure, par région, par mois et par année, en MW |
| `qualite_statuts.csv` | Part des données définitives, consolidées, redressées… par année |
| `qualite_validation.csv` | Écart quotidien entre la somme des régions et le national |
| `journal_nettoyage.csv` | Chaque opération de nettoyage et le nombre de lignes concernées |

**Décisions prises à partir du contrôle de l'étape 1 :**

| Problème | Traitement |
|---|---|
| Doublons au passage à l'heure d'été (fin mars) | Suppression des lignes « 02:00 » et « 02:30 » : cette heure n'existe pas ce jour-là |
| Électricité négative (1 valeur) | Remplacée par une valeur manquante |
| Statut Teréga « vers » (1 ligne) | Remplacé par une valeur manquante |
| `flag_ignore` vaut toujours « non » | Colonne supprimée |
| `date_heure` est en UTC | On utilise `date` + `heure`, en heure locale |
| Gaz horaire, électricité demi-horaire | Énergie : gaz MWh = somme des valeurs horaires ; électricité MWh = somme des valeurs demi-horaires × 0,5 |
| Jours incomplets | Signalés par une colonne `complet`, pour pouvoir les exclure des comparaisons |

In [1]:
import re
import unicodedata
from pathlib import Path

import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

DOSSIER = Path.cwd()
FICHIER_REGIONAL = DOSSIER / "conso_regionale.csv"
FICHIER_NATIONAL = DOSSIER / "conso_nationale.csv"

journal = []   # trace de chaque opération de nettoyage


def noter(etape, lignes_concernees, detail=""):
    journal.append({"etape": etape, "lignes_concernees": int(lignes_concernees), "detail": detail})
    print(f"- {etape} : {lignes_concernees:,} ligne(s). {detail}")

## 1. Lecture et renommage des colonnes

Même renommage qu'à l'étape 1 : chaque libellé d'origine reçoit un nom court, comme `gaz_total_mw` ou `elec_rte_mw`.

In [2]:
OPERATEURS = ["natran", "grtgaz", "terega", "rte"]


def normaliser(texte):
    texte = unicodedata.normalize("NFKD", texte).encode("ascii", "ignore").decode()
    return re.sub(r"[^a-z0-9]+", "_", texte.lower()).strip("_")


def nom_court(colonne):
    n = normaliser(colonne)
    op = next((o for o in OPERATEURS if o in n), "")
    if "date" in n and "heure" in n:
        return "date_heure"
    if n in ("date", "heure", "region", "flag_ignore"):
        return n
    if "code" in n and "region" in n:
        return "code_region"
    if "statut" in n:
        return f"statut_{op}"
    if "gaz" in n:
        return "gaz_total_mw" if "totale" in n else f"gaz_{op}_mw"
    if "electricite" in n:
        return f"elec_{op}_mw"
    if "totale" in n:
        return "total_mw"
    return n


def lire_csv(chemin):
    with open(chemin, encoding="utf-8") as f:
        entete = f.readline()
    sep = ";" if entete.count(";") > entete.count(",") else ","
    df = pd.read_csv(chemin, sep=sep, low_memory=False)
    return df.rename(columns={c: nom_court(c) for c in df.columns})


def en_date(serie):
    # Les dates sont au format 2013-01-31 (régional) ou 31/01/2013 (national)
    iso = pd.to_datetime(serie, format="%Y-%m-%d", errors="coerce")
    fr = pd.to_datetime(serie, format="%d/%m/%Y", errors="coerce")
    return iso.fillna(fr)


reg = lire_csv(FICHIER_REGIONAL)
nat = lire_csv(FICHIER_NATIONAL)
noter("Lecture du fichier régional", len(reg))
noter("Lecture du fichier national", len(nat))

- Lecture du fichier régional : 2,874,807 ligne(s). 
- Lecture du fichier national : 257,136 ligne(s). 


## 2. Passage à l'heure d'été : suppression des lignes fantômes

Le dernier dimanche de mars, on passe directement de 2 h à 3 h. Le fichier contient pourtant des lignes « 02:00 » et « 02:30 », rattachées à la même heure UTC que « 03:00 » et « 03:30 ».

On garde la ligne dont l'heure locale existe vraiment (03:00) et on supprime la ligne fantôme (02:00). Sans cette correction, ces journées compteraient 24 heures de gaz au lieu de 23.

In [13]:
def supprimer_heures_fantomes(df, nom):
    cle = ["date_heure", "region"] if "region" in df.columns else ["date_heure"]
    en_double = df.duplicated(subset=cle, keep=False)
    fantomes = en_double & df["heure"].isin(["02:00", "02:30"])
    noter(f"Heures fantômes supprimées ({nom})", fantomes.sum(),
          f"jours concernés : {df.loc[fantomes, 'date'].nunique()}")
    df = df[~fantomes].copy()
    restants = df.duplicated(subset=cle).sum()
    noter(f"Doublons restants ({nom})", restants)
    df = df.drop_duplicates(subset=cle, keep="first")
    noter(f"Doublons restants supprimés ({nom})", restants)
    return df


reg = supprimer_heures_fantomes(reg, "régional")
nat = supprimer_heures_fantomes(nat, "national")

- Heures fantômes supprimées (régional) : 0 ligne(s). jours concernés : 0
- Doublons restants (régional) : 0 ligne(s). 
- Doublons restants supprimés (régional) : 0 ligne(s). 
- Heures fantômes supprimées (national) : 0 ligne(s). jours concernés : 0
- Doublons restants (national) : 3 ligne(s). 
- Doublons restants supprimés (national) : 3 ligne(s). 


## 3. Valeurs aberrantes et colonnes inutiles

Une consommation ne peut pas être négative. Le statut « vers » ne fait pas partie des valeurs possibles. Enfin, `flag_ignore` vaut « non » sur toutes les lignes, donc elle n'apporte aucune information.

In [14]:
negatifs = reg["elec_rte_mw"] < 0
noter("Électricité négative remplacée par une valeur manquante", negatifs.sum(),
      "; ".join(reg.loc[negatifs, "region"] + " " + reg.loc[negatifs, "date"] + " " + reg.loc[negatifs, "heure"]))
reg.loc[negatifs, "elec_rte_mw"] = pd.NA

statuts_valides = {"Définitif", "Définitive", "Consolidé", "Intermédiaire", "Redressé", "Meilleur Statut"}
for df, nom in [(reg, "régional"), (nat, "national")]:
    for col in [c for c in df.columns if c.startswith("statut_")]:
        invalides = df[col].notna() & ~df[col].isin(statuts_valides)
        if invalides.any():
            noter(f"Statut invalide remplacé ({nom}, {col})", invalides.sum(),
                  f"valeurs : {sorted(df.loc[invalides, col].unique())}")
            df.loc[invalides, col] = pd.NA
        # Harmonisation de l'orthographe
        df[col] = df[col].replace({"Définitive": "Définitif"})

reg = reg.drop(columns=["flag_ignore"], errors="ignore")
nat = nat.drop(columns=["flag_ignore"], errors="ignore")
noter("Colonne flag_ignore supprimée (toujours « non »)", 0)

- Électricité négative remplacée par une valeur manquante : 0 ligne(s). 
- Colonne flag_ignore supprimée (toujours « non ») : 0 ligne(s). 


## 4. Statuts des données, pour la section « Qualité »

Pour chaque année, on calcule la part des valeurs de gaz (NaTran) et d'électricité (RTE) selon leur statut :
- **Définitif** : la valeur ne changera plus ;
- **Consolidé** ou **Intermédiaire** : la valeur peut encore être révisée ;
- **Redressé** : la valeur a été corrigée par le producteur.

In [15]:
reg["annee"] = reg["date"].str[:4].astype(int)

tables_statuts = []
for col, source in [("statut_natran", "Gaz NaTran"), ("statut_terega", "Gaz Teréga"), ("statut_rte", "Électricité RTE")]:
    t = (reg.dropna(subset=[col]).groupby(["annee", col]).size()
         .rename("lignes").reset_index().rename(columns={col: "statut"}))
    t["source"] = source
    t["pct"] = (100 * t["lignes"] / t.groupby("annee")["lignes"].transform("sum")).round(1)
    tables_statuts.append(t)

qualite_statuts = pd.concat(tables_statuts, ignore_index=True)[["source", "annee", "statut", "lignes", "pct"]]
qualite_statuts.pivot_table(index="annee", columns=["source", "statut"], values="pct").fillna(0)

source Gaz NaTran          Gaz Teréga               Électricité RTE          
statut  Définitif Redressé  Définitif Intermédiaire       Consolidé Définitif
annee                                                                        
2013        100.0      0.0      100.0           0.0             0.0     100.0
2014        100.0      0.0      100.0           0.0             0.0     100.0
2015        100.0      0.0      100.0           0.0             0.0     100.0
2016        100.0      0.0      100.0           0.0             0.0     100.0
2017        100.0      0.0      100.0           0.0             0.0     100.0
2018        100.0      0.0      100.0           0.0             0.0     100.0
2019        100.0      0.0      100.0           0.0             0.0     100.0
2020        100.0      0.0      100.0           0.0           100.0       0.0
2021        100.0      0.0      100.0           0.0           100.0       0.0
2022          0.0    100.0      100.0           0.0           100.0       0.0
2023        100.0      0.0      100.0           0.0           100.0       0.0
2024        100.0      0.0      100.0           0.0           100.0       0.0
2025        100.0      0.0      100.0           0.0           100.0       0.0
2026        100.0      0.0       87.2          12.8           100.0       0.0

## 5. Des puissances (MW) aux énergies (MWh)

Les valeurs sont des **puissances moyennes** sur la période de mesure :
- le gaz est mesuré **chaque heure** : une valeur en MW sur 1 heure donne la même valeur en MWh ;
- l'électricité est mesurée **chaque demi-heure** : une valeur en MW sur 0,5 heure donne valeur × 0,5 MWh.

On travaille en heure **locale** (`date` + `heure`), parce que les habitudes de consommation suivent l'heure française.

In [16]:
reg["jour"] = pd.to_datetime(reg["date"], format="%Y-%m-%d")
reg["heure_num"] = reg["heure"].str[:2].astype(int)
reg["heure_pleine"] = reg["heure"].str.endswith(":00")

reg["gaz_mwh"] = reg["gaz_total_mw"]            # 1 heure
reg["elec_mwh"] = reg["elec_rte_mw"] * 0.5      # 1 demi-heure

## 6. Table principale : consommation par jour et par région

Pour chaque région et chaque jour, on calcule :
- `gaz_gwh` et `elec_gwh` : l'énergie consommée dans la journée (1 GWh = 1 000 MWh) ;
- `heures_gaz` : le nombre d'heures où le gaz est renseigné, comparé au nombre d'heures que compte réellement la journée (23, 24 ou 25 heures selon le changement d'heure) ;
- `complet` : vrai si le gaz et l'électricité sont renseignés pour toutes les heures de la journée.

On ajoute aussi des variables utiles pour l'analyse : année, mois, jour de la semaine, et **hiver de chauffe** (d'octobre à mars, par exemple « 2022-2023 »).

In [17]:
jour = (reg.groupby(["jour", "code_region", "region"])
        .agg(gaz_mwh=("gaz_mwh", "sum"),
             elec_mwh=("elec_mwh", "sum"),
             heures_gaz=("gaz_mwh", "count"),
             heures_attendues=("heure_pleine", "sum"),
             demi_heures_elec=("elec_mwh", "count"),
             demi_heures_attendues=("heure", "size"))
        .reset_index())

# Une journée sans aucune valeur donne une somme vide (NaN) et non 0
jour["gaz_gwh"] = (jour.pop("gaz_mwh") / 1000).round(3).where(jour["heures_gaz"] > 0)
jour["elec_gwh"] = (jour.pop("elec_mwh") / 1000).round(3).where(jour["demi_heures_elec"] > 0)
jour["complet"] = ((jour["heures_gaz"] == jour["heures_attendues"])
                   & (jour["demi_heures_elec"] == jour["demi_heures_attendues"])
                   & (jour["heures_attendues"] >= 23))

# Variables calendaires
jour["annee"] = jour["jour"].dt.year
jour["mois"] = jour["jour"].dt.month
jour["jour_semaine"] = jour["jour"].dt.dayofweek          # 0 = lundi
debut_hiver = jour["annee"].where(jour["mois"] >= 10, jour["annee"] - 1)
jour["hiver"] = (debut_hiver.astype(str) + "-" + (debut_hiver + 1).astype(str)).where(
    jour["mois"].isin([10, 11, 12, 1, 2, 3]))

incomplets = ~jour["complet"]
noter("Journées région incomplètes (signalées, non supprimées)", incomplets.sum(),
      f"sur {len(jour):,} journées région")
jour[incomplets].groupby("annee").size().rename("journees_incompletes")

- Journées région incomplètes (signalées, non supprimées) : 30 ligne(s). sur 59,892 journées région


annee
2013    12
2020    12
2022     5
2025     1
Name: journees_incompletes, dtype: int64

## 7. Profil horaire : à quelle heure consomme-t-on ?

Pour chaque région, année, mois et heure, on calcule la puissance moyenne en MW. Cette table sert aux graphiques de saisonnalité et de pointe du matin et du soir.

In [18]:
profil_horaire = (reg.groupby(["code_region", "region", "annee", reg["jour"].dt.month.rename("mois"), "heure_num"])
                  .agg(gaz_mw=("gaz_total_mw", "mean"), elec_mw=("elec_rte_mw", "mean"))
                  .round(1).reset_index().rename(columns={"heure_num": "heure"}))
print(f"{len(profil_horaire):,} lignes")
profil_horaire.head()

47,232 lignes


,code_region,region,annee,mois,heure,gaz_mw,elec_mw
0,11,Île-de-France,2013,1,0,3537.1,10942.1
1,11,Île-de-France,2013,1,1,3539.7,10040.2
2,11,Île-de-France,2013,1,2,3561.8,9459.3
3,11,Île-de-France,2013,1,3,3582.0,9093.7
4,11,Île-de-France,2013,1,4,3630.0,8904.8


## 8. Validation après nettoyage : somme des régions = national ?

On refait la comparaison de l'étape 1 sur les données nettoyées, jour par jour. Les jours où l'écart dépasse 1 % sont signalés.

In [19]:
nat["jour"] = en_date(nat["date"])
nat_jour = nat.groupby("jour").agg(
    gaz_national_gwh=("gaz_total_mw", lambda s: s.sum(min_count=1) / 1000),
    elec_national_gwh=("elec_rte_mw", lambda s: s.sum(min_count=1) * 0.5 / 1000))

somme = jour.groupby("jour")[["gaz_gwh", "elec_gwh"]].sum(min_count=1)
qualite_validation = somme.join(nat_jour, how="inner")
qualite_validation["ecart_gaz_pct"] = (100 * (qualite_validation["gaz_gwh"] / qualite_validation["gaz_national_gwh"] - 1)).round(3)
qualite_validation["ecart_elec_pct"] = (100 * (qualite_validation["elec_gwh"] / qualite_validation["elec_national_gwh"] - 1)).round(3)
qualite_validation = qualite_validation.round(3).reset_index()

for col in ["ecart_gaz_pct", "ecart_elec_pct"]:
    hors = qualite_validation[col].abs() > 1
    noter(f"Jours avec un écart > 1 % ({col})", hors.sum(),
          f"écart médian : {qualite_validation[col].median():.3f} %")

qualite_validation.loc[qualite_validation["ecart_gaz_pct"].abs() > 1,
                       ["jour", "gaz_gwh", "gaz_national_gwh", "ecart_gaz_pct"]].head(15)

- Jours avec un écart > 1 % (ecart_gaz_pct) : 1,139 ligne(s). écart médian : 0.000 %
- Jours avec un écart > 1 % (ecart_elec_pct) : 24 ligne(s). écart médian : 0.000 %


,jour,gaz_gwh,gaz_national_gwh,ecart_gaz_pct
0,2013-01-01,1084.490,1658.511,-34.611
1,2013-01-02,1639.959,1947.763,-15.803
2,2013-01-03,1664.405,1936.286,-14.041
3,2013-01-04,1588.131,1844.629,-13.905
4,2013-01-05,1482.078,1732.800,-14.469
5,2013-01-06,1482.236,1744.093,-15.014
6,2013-01-07,1781.519,2067.906,-13.849
7,2013-01-08,1900.251,2221.277,-14.452
8,2013-01-09,1938.468,2276.972,-14.866
9,2013-01-10,1867.238,2179.156,-14.314


In [20]:
v = qualite_validation.assign(mois=pd.to_datetime(qualite_validation["jour"]).dt.to_period("M"))
diag = v.groupby("mois").agg(
    jours=("jour", "size"),
    jours_ecart_gaz_sup_1pct=("ecart_gaz_pct", lambda s: (s.abs() > 1).sum()),
    ecart_gaz_median=("ecart_gaz_pct", "median"),
    jours_ecart_elec_sup_1pct=("ecart_elec_pct", lambda s: (s.abs() > 1).sum()))
diag[(diag["jours_ecart_gaz_sup_1pct"] > 0) | (diag["jours_ecart_elec_sup_1pct"] > 0)]

,jours,jours_ecart_gaz_sup_1pct,ecart_gaz_median,jours_ecart_elec_sup_1pct
mois,,,,
2013-01,31,31,-15.8030,1
2013-02,28,28,-15.9900,0
2013-03,31,31,-15.9030,0
2013-04,30,30,-14.1485,0
2013-05,31,31,-12.5060,0
2013-06,30,1,-0.0395,0
2016-04,30,0,-0.0020,3
2017-03,31,30,-3.3230,0
2017-04,30,30,1.9665,0


## 9. Premier aperçu : consommation annuelle de gaz

Ce n'est pas encore l'analyse, seulement un contrôle de vraisemblance. On ne garde que les années complètes et les journées complètes. Le total doit être de l'ordre de quelques centaines de TWh par an.

In [21]:
annuel = (jour[jour["complet"] & jour["annee"].between(2013, 2025)]
          .groupby("annee")[["gaz_gwh", "elec_gwh"]].sum() / 1000).round(1)
annuel.columns = ["gaz_twh", "elec_twh"]
annuel["gaz_vs_2018_2021_pct"] = (100 * (annuel["gaz_twh"] / annuel.loc[2018:2021, "gaz_twh"].mean() - 1)).round(1)
annuel

,gaz_twh,elec_twh,gaz_vs_2018_2021_pct
annee,,,
2013,454.6,490.7,-2.5
2014,416.8,462.3,-10.6
2015,449.7,473.1,-3.5
2016,491.2,480.3,5.4
2017,493.7,479.0,5.9
2018,470.3,475.5,0.9
2019,478.8,470.4,2.7
2020,443.1,444.5,-4.9
2021,471.9,467.1,1.3


## 10. Enregistrement des tables

In [22]:
colonnes_jour = ["jour", "annee", "mois", "jour_semaine", "hiver", "code_region", "region",
                 "gaz_gwh", "elec_gwh", "heures_gaz", "heures_attendues",
                 "demi_heures_elec", "demi_heures_attendues", "complet"]
sorties = {
    "conso_jour_region.csv": jour[colonnes_jour],
    "profil_horaire.csv": profil_horaire,
    "qualite_statuts.csv": qualite_statuts,
    "qualite_validation.csv": qualite_validation,
    "journal_nettoyage.csv": pd.DataFrame(journal),
}
for nom, table in sorties.items():
    table.to_csv(DOSSIER / nom, index=False, encoding="utf-8")
    print(f"{nom:28s} {len(table):>8,} lignes  {(DOSSIER / nom).stat().st_size / 1e6:6.2f} Mo")

pd.DataFrame(journal)

conso_jour_region.csv          59,892 lignes    4.67 Mo
profil_horaire.csv             47,232 lignes    2.10 Mo
qualite_statuts.csv                43 lignes    0.00 Mo
qualite_validation.csv          4,991 lignes    0.29 Mo
journal_nettoyage.csv              23 lignes    0.00 Mo


,etape,lignes_concernees,detail
0,Lecture du fichier régional,2874807,
1,Lecture du fichier national,257136,
2,Heures fantômes supprimées (régional),336,jours concernés : 14
3,Doublons restants (régional),0,
4,Heures fantômes supprimées (national),30,jours concernés : 15
5,Doublons restants (national),3,
6,Électricité négative remplacée par une valeur ...,1,Auvergne-Rhône-Alpes 2025-11-13 11:30
7,"Statut invalide remplacé (régional, statut_ter...",1,valeurs : ['vers']
8,Colonne flag_ignore supprimée (toujours « non »),0,
9,"Journées région incomplètes (signalées, non su...",30,"sur 59,892 journées région"
